In [2]:
!pip install hopsworks

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.6/258.6 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 8.3 MB/s eta 0:00:00
  Created 

In [3]:
!pip install confluent-kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 67.8 MB/s eta 0:00:00


In [4]:
from google.colab import files
uploaded = files.upload()  # apni feature_df.csv choose karo

Saving feature_df.csv to feature_df.csv


In [5]:
import getpass
HOPSWORKS_API_KEY = getpass.getpass("Enter your Hopsworks API key: ")
HOPSWORKS_PROJECT_NAME = input("Enter your Hopsworks project name: ")


Enter your Hopsworks API key: ··········
Enter your Hopsworks project name: practice_project


In [8]:
import hopsworks
import pandas as pd

df = pd.read_csv("feature_df.csv")
df["collection_timestamp"] = pd.to_datetime(df["collection_timestamp"], utc=True).dt.tz_localize(None)
if "timestamp" in df.columns:
    df = df.drop(columns=["timestamp"])

project = hopsworks.login(project=HOPSWORKS_PROJECT_NAME, api_key_value=HOPSWORKS_API_KEY)
fs = project.get_feature_store()

fg = fs.get_or_create_feature_group(
    name="aqi_features",
    version=2,   # <-- v1 se v2 (feels_like fix + correct rolling/horizon windows)
    description=(
        "Engineered AQI + weather features (Phase 2 output). "
        "v2: fixes feels_like (was null for all historical records) and "
        "rolling-window/forecast-horizon sizes (were 8h/16h/24h-ahead, "
        "now correctly 24h/48h/72h-ahead at the pipeline's actual hourly interval)."
    ),
    primary_key=["city", "collection_timestamp"],
    event_time="collection_timestamp",
    online_enabled=False,
    time_travel_format="HUDI",
)

fg.insert(df)
print(f"Inserted {len(df)} rows into v2.")


Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41109


Uploading Dataframe: 100.00% |██████████| Rows 5887/5887 | Elapsed Time: 00:03 | Remaining Time: 00:00


Inserted 5887 rows into v2.


Use fg.materialization_job.run(args=-op offline_fg_materialization -path hdfs:///Projects/practice_project/Resources/jobs/aqi_features_2_offline_fg_materialization/config_1785270686318) to trigger the materialization job again.


In [9]:
import time

max_wait_seconds = 300  # 5 minutes
interval_seconds = 15
elapsed = 0
result = None

while elapsed < max_wait_seconds:
    try:
        result = fg.read()
        break
    except Exception as e:
        print(f"Materialization job not finished yet ({elapsed}s elapsed) — retrying in {interval_seconds}s...")
        time.sleep(interval_seconds)
        elapsed += interval_seconds

if result is not None:
    print(f"Read back {len(result)} rows.")
    print(result.head())
else:
    print("Still not ready after 5 minutes — check the job execution link in the Hopsworks UI for errors.")

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (2.54s) 
Read back 5887 rows.
       collection_timestamp    city   latitude  longitude  temperature  \
0 2026-04-19 02:00:00+00:00  Sukkur  27.732864  68.865166         26.1   
1 2026-07-08 05:00:00+00:00  Sukkur  27.732864  68.865166         36.0   
2 2026-01-10 16:00:00+00:00  Sukkur  27.732864  68.865166         11.4   
3 2026-07-08 12:00:00+00:00  Sukkur  27.732864  68.865166         41.2   
4 2026-04-14 14:00:00+00:00  Sukkur  27.732864  68.865166         31.4   

   feels_like  humidity  pressure  wind_speed  wind_direction  ...  \
0        26.4        57     998.8        14.0              78  ...   
1        40.3        49     988.2        11.8             141  ...   
2        10.4        73    1013.9         0.7              14  ...   
3        43.3        30     984.3        12.5             155  ...   
4        32.0        34     996.4         3.2             142  ...   

   pm2_5_rolling_mean_24h  

In [ ]:
df["pm10"].isna().sum()

np.int64(1936)